In [ ]:
import sys
print(sys.version)

In [ ]:
!pip install huggingface_hub
!pip install sentencepiece
!pip install protobuf
!pip install -U bitsandbytes

In [ ]:
hf_token = open('conf/hf_token.txt').read()

In [ ]:
import json, torch, re
from huggingface_hub import login
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, pipeline

model_id = "mistralai/Mistral-7B-Instruct-v0.3"
tokenizer = AutoTokenizer.from_pretrained(model_id)
quantization_config = BitsAndBytesConfig(load_in_8bit=True)

model = AutoModelForCausalLM.from_pretrained(
    model_id,
    # load_in_8bit=True,
    quantization_config = quantization_config,
    torch_dtype=torch.float16,
    device_map="auto",
)

In [ ]:
# Test the model
prompt = """[INST]Translate the phrase "How are you?" to German. Please place the answer of the question in a JSON object with the answer in \"answer\". [/INST]"""
input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")
outputs = model.generate(**input_ids, max_new_tokens=100, pad_token_id=tokenizer.eos_token_id, temperature=0.7, do_sample = True)
print(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
def get_response(prompt):
    prompt_inst = f'[INST] {prompt} [/INST]'
    input_ids = tokenizer(prompt, return_tensors="pt").to("cuda")
    outputs = model.generate(**input_ids, 
                             max_new_tokens=500, 
                             pad_token_id=tokenizer.eos_token_id, 
                             temperature=0.3, 
                             do_sample=True)
    return(tokenizer.decode(outputs[0], skip_special_tokens=True))

In [ ]:
def extract_json_from_string(text):
    # This regex matches a JSON object or array
    json_pattern = r'({.*?})|(\[.*?\])' 
    matches = re.findall(json_pattern, text, re.DOTALL)
    for match in matches:
        json_str = match[0] if match[0] else match[1]
        try:
            return json.loads(json_str)
        except json.JSONDecodeError:
            continue
    
    return None


In [ ]:
def inject_noise_with_mistral(original_sentence):
    prompt = f"""Take the sentence "{original_sentence}"
    For data privacy purposes, you need to change some details in that sentence to thwart 
    re-identification by replacing original details with similarly-specific ones.
    Please place the modified text in a JSON object with the answer in "noisy".
    Put nothing else in the JSON."""
    response = get_response(prompt)
    try:
        response_json = extract_json_from_string(response)
        return response_json['noisy']
    except:
        return f'There was an issue with json. Raw response:\n{response}'

In [ ]:
def genericize_with_mistral(original_sentence):
    prompt = f"""Take the sentence "{original_sentence}"
    For data privacy purposes, you need to generalize some details in that sentence to thwart
    re-identification by replacing original details with less-specific ones.
    Please place the modified text in a JSON object with the answer in "noisy".
    Put nothing else in the JSON."""
    response = get_response(prompt)
    try:
        response_json = extract_json_from_string(response)
        return response_json['noisy']
    except:
        return f'There was an issue with json. Raw response:\n{response}'

In [ ]:
sentence = 'A data privacy speaker ate four tubs of Rocky Road ice cream last week after blowing a call while refereeing a rugby match.'

print('Noisy:\n' + inject_noise_with_mistral(sentence))
print('---------------------------------------')
print('Generic:\n' + genericize_with_mistral(sentence))